# 06 — Comparação entre Modelos

Objetivo: avaliar os 3 modelos (Regressão Logística, Random Forest, SVM) no **mesmo conjunto de teste (2023–2024)** e justificar a escolha do modelo final.

## Carregamento e Preparação dos Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, mean_absolute_error, mean_squared_error,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc
)

FEATURES = ['Plantel', 'Estrangeiros', 'Valor de Mercado Total']
TARGET   = 'Status_bin'

df = pd.read_excel(os.path.join('..','dados','BASE_FINAL.xlsx'), sheet_name='CLUBES')
df.columns = df.columns.str.strip()
df[TARGET] = df['Situacao'].apply(lambda x: 0 if str(x).strip().lower()=='rebaixado' else 1)

df_rot = df[df['Temporada'] < 2025].copy()
df_tr  = df_rot[df_rot['Temporada'] <= 2022]
df_te  = df_rot[df_rot['Temporada']  > 2022]

scaler = StandardScaler()
X_tr = scaler.fit_transform(df_tr[FEATURES]); y_tr = df_tr[TARGET].values
X_te = scaler.transform(df_te[FEATURES]);     y_te = df_te[TARGET].values

modelos = {
    'Regressao Logistica': LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    'SVM':                 SVC(probability=True, random_state=42, class_weight='balanced')
}

for nome, m in modelos.items():
    m.fit(X_tr, y_tr)
    print(f'OK: {nome} treinado.')

## Tabela Comparativa de Métricas

In [ ]:
resultados = []
for nome, m in modelos.items():
    y_pred = m.predict(X_te)
    y_prob = m.predict_proba(X_te)[:,0]  # prob classe 0 (Rebaixado)
    fpr, tpr, _ = roc_curve(y_te, y_prob, pos_label=0)
    roc_auc = auc(fpr, tpr)
    resultados.append({
        'Modelo': nome,
        'Acuracia': round(accuracy_score(y_te, y_pred), 4),
        'MAE':      round(mean_absolute_error(y_te, y_pred), 4),
        'RMSE':     round(float(np.sqrt(mean_squared_error(y_te, y_pred))), 4),
        'AUC':      round(roc_auc, 4)
    })

df_res = pd.DataFrame(resultados).set_index('Modelo')
print('Metricas no Conjunto de Teste (2023-2024):')
print(df_res.to_string())
df_res.style.format('{:.4f}').background_gradient(cmap='Greens', subset=['Acuracia','AUC']).background_gradient(cmap='Reds_r', subset=['MAE','RMSE'])

## Gráfico Comparativo de Métricas

In [ ]:
df_plot = df_res.reset_index().melt(id_vars='Modelo', var_name='Metrica', value_name='Valor')
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Comparacao de Metricas — Conjunto de Teste', fontsize=13, fontweight='bold')
cores = ['#2980b9', '#27ae60', '#8e44ad']
for ax, metrica in zip(axes, ['Acuracia','MAE','RMSE','AUC']):
    sub = df_plot[df_plot['Metrica']==metrica]
    bars = ax.bar(sub['Modelo'], sub['Valor'], color=cores, edgecolor='white', width=0.5)
    ax.set_title(metrica, fontweight='bold')
    ax.set_xticklabels(sub['Modelo'], rotation=20, ha='right', fontsize=8)
    for bar in bars:
        ax.annotate(f'{bar.get_height():.3f}', (bar.get_x()+bar.get_width()/2, bar.get_height()), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
os.makedirs(os.path.join('..','img'), exist_ok=True)
plt.savefig(os.path.join('..','img','comparacao_metricas.png'), dpi=150, bbox_inches='tight')
plt.show()

## Curvas ROC dos 3 Modelos

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
cores_roc = {'Regressao Logistica': '#2980b9', 'Random Forest': '#27ae60', 'SVM': '#8e44ad'}
for nome, m in modelos.items():
    y_prob = m.predict_proba(X_te)[:,0]
    fpr, tpr, _ = roc_curve(y_te, y_prob, pos_label=0)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{nome} (AUC={roc_auc:.3f})', color=cores_roc[nome], lw=2)
ax.plot([0,1],[0,1],'k--', alpha=0.5, label='Aleatório')
ax.set_xlabel('Taxa de Falso Positivo'); ax.set_ylabel('Taxa de Verdadeiro Positivo')
ax.set_title('Curvas ROC — Comparação dos 3 Modelos', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join('..','img','roc_comparacao.png'), dpi=150)
plt.show()

## Matrizes de Confusão dos 3 Modelos

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Matrizes de Confusao — Conjunto de Teste (2023-2024)', fontsize=13, fontweight='bold')
for ax, (nome, m) in zip(axes, modelos.items()):
    y_pred = m.predict(X_te)
    cm = confusion_matrix(y_te, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Rebaixado','Permaneceu'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    acc = accuracy_score(y_te, y_pred)
    ax.set_title(f'{nome}\nAcuracia: {acc:.2%}', fontweight='bold', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join('..','img','matrizes_confusao.png'), dpi=150, bbox_inches='tight')
plt.show()

## Justificativa da Escolha do Modelo Final

A **Regressão Logística** foi escolhida como modelo final pelos seguintes motivos:
1. **Interpretabilidade**: os coeficientes revelam diretamente a influência de cada feature
2. **Desempenho competitivo** no conjunto de teste
3. **Adequação ao problema**: variável binária, poucas features numéricas
4. **Comunicabilidade**: resultados facilmente explicáveis para não-especialistas

## Discussão: Desbalanceamento de Classes

O dataset apresenta **desbalanceamento significativo**: apenas ~20% dos registros são rebaixamentos (44 de 220). Sem correção, o modelo tende a prever sempre "Permaneceu", obtendo alta acurácia bruta mas zero utilidade prática.

**Solução adotada:** `class_weight='balanced'`, que pondera automaticamente as classes inversamente proporcional às suas frequências, forçando o modelo a prestar mais atenção aos casos de rebaixamento.